# CMSC 173 &middot; Machine Learning &mdash; Week 8 Lab
## PCA: Compressing Data by Finding Its Main Directions

Many features are secretly redundant &mdash; they're just noisy copies of a few hidden factors.
**Principal Component Analysis** finds the directions your data varies along most, so you can
describe it with fewer numbers. You'll build PCA **from scratch** &mdash; centre, covariance,
eigenvectors &mdash; then read a **scree plot** and see 5 features squashed into a 2-D picture.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib (from scratch; eigen-decomposition via numpy).** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + a secretly-2D dataset

We build 5 features out of only **2 hidden factors** plus noise. So the data *looks* 5-dimensional
but really lives on a 2-D sheet &mdash; exactly what PCA should discover.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(173)

n = 300
f1, f2 = rng.normal(0,1,n), rng.normal(0,1,n)        # two hidden factors
X = np.column_stack([
    3.0*f1 + 0.3*rng.normal(0,1,n),                  # features 1-2 driven by factor 1
    2.0*f1 + 0.3*rng.normal(0,1,n),
    2.5*f2 + 0.3*rng.normal(0,1,n),                  # features 3-4 driven by factor 2
    1.5*f2 + 0.3*rng.normal(0,1,n),
    1.2*f1 - 0.8*f2 + 0.3*rng.normal(0,1,n),         # feature 5: a MIX of both factors
])
print('data shape:', X.shape, '(300 rows, 5 features)')

**Reading the code:** features 1&ndash;2 are scaled copies of factor 1, features 3&ndash;4 of factor 2, and
feature 5 is a *mix* of both. Every column is built from just **2** underlying factors, so there
are really only ~2 independent directions here. PCA doesn't know that &mdash; let's see if it finds it.

---
## Part 1 &middot; Step 1: centre (and scale) the data

PCA is about *variation*, which is measured from the mean &mdash; so first we subtract each feature's
mean. We also divide by the standard deviation so features on bigger scales don't automatically
look 'more important'.

In [ ]:
Xc = (X - X.mean(axis=0)) / X.std(axis=0)            # standardise: each feature mean 0, std 1
print('column means after centring:', np.round(Xc.mean(axis=0), 3))
print('column stds   after scaling :', np.round(Xc.std(axis=0), 3))

**Reading the code:** `X.mean(axis=0)` is the mean of *each column*; subtracting it centres every
feature on zero, and dividing by `X.std(axis=0)` rescales them. The printouts confirm every column
now has mean 0 and std 1 &mdash; a fair starting line for all features.

**Answer here:**

1. Why must we centre before looking at directions of variation? What would a *non*-zero mean do to
   the 'direction the data spreads'?
   &rarr; *your answer*

---
## Part 2 &middot; Step 2: the covariance matrix

The **covariance matrix** summarises how every pair of features moves together. For centred data
it's just $\Sigma = \frac{1}{n} X^\top X$ &mdash; a 5&times;5 table. Big off-diagonal numbers mean two
features carry the same information.

In [ ]:
cov = (Xc.T @ Xc) / len(Xc)                          # 5x5 covariance matrix
print(np.round(cov, 2))

**Reading the code:** `Xc.T @ Xc` sums, for every pair of features, the product of their values;
dividing by `n` averages it. The diagonal is each feature's variance (all ~1 after scaling); the
large off-diagonal blocks (features 1&ndash;2 together, 3&ndash;4 together) reveal the redundancy we built in.

---
## Part 3 &middot; Step 3: eigenvectors = the principal directions

The magic step. The **eigenvectors** of the covariance matrix are the axes the data varies along;
each **eigenvalue** says how much variance lies along its axis. We compute them, then sort from
most to least variance.

In [ ]:
eigvals, eigvecs = np.linalg.eigh(cov)               # (1) eigen-decompose (symmetric -> eigh)
order = np.argsort(eigvals)[::-1]                     # (2) sort biggest-variance first
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

print('eigenvalues (variance per direction):')
print(np.round(eigvals, 3))

**Reading the code, line by line:**
- **(1)** `np.linalg.eigh` returns eigenvalues/vectors of a symmetric matrix (covariance always is).
- **(2)** it returns them smallest-first, so `argsort(...)[::-1]` flips to largest-first and we
  reorder both. The first two eigenvalues are much larger than the rest &mdash; the data's variance is
  concentrated in **2 directions**, just as we designed. Those two eigenvectors are PC1 and PC2.

**Answer here:**

1. Look at the eigenvalues: how many are clearly large before they drop off? Does that match the
   number of hidden factors we used?
   &rarr; *your answer*

---
## Part 4 &middot; The scree plot: how many components to keep

Divide each eigenvalue by the total to get the **fraction of variance** it explains. Plotting these
(a **scree plot**) shows where the 'elbow' is &mdash; how many components are worth keeping.

In [ ]:
explained = eigvals / eigvals.sum()                  # fraction of variance per component
cumulative = np.cumsum(explained)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(range(1, 6), explained); ax[0].set_xlabel('component'); ax[0].set_ylabel('variance explained')
ax[0].set_title('Scree plot (elbow after 2)')
ax[1].plot(range(1, 6), cumulative, 'o-'); ax[1].axhline(0.9, ls='--', color='gray')
ax[1].set_xlabel('components kept'); ax[1].set_ylabel('cumulative variance')
ax[1].set_title('Cumulative variance'); plt.tight_layout(); plt.show()
print('variance explained:', np.round(explained, 3))
print(f'first 2 components keep {cumulative[1]:.1%} of all the variance')

**Reading the code:** `explained` is each component's share of the total variance; `cumsum` adds
them up. The bar plot drops sharply after component 2 (the **elbow**), and the cumulative plot
shows 2 components already cover ~90%. Conclusion: keep 2, discard 3 &mdash; a 5&rarr;2 compression with
almost no information lost.

**Answer here:**

1. If two components keep ~90% of the variance, what have you *gained* by dropping to 2 features,
   and what (small thing) have you *lost*?
   &rarr; *your answer*

---
## Part 5 &middot; Project the data into 2-D and look at it

Finally, use it: multiply the centred data by the first two eigenvectors to get each point's
coordinates in the new 2-D space. Five columns become two, and we can actually plot it.

In [ ]:
Z = Xc @ eigvecs[:, :2]                              # (1) project onto PC1 and PC2
print('projected shape:', Z.shape, '(300 rows, 2 columns now)')

plt.figure(figsize=(6,5))
plt.scatter(Z[:,0], Z[:,1], alpha=0.6, c=f1, cmap='viridis')  # colour by hidden factor 1
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('5-D data drawn in 2-D (colour = hidden factor 1)')
plt.colorbar(label='factor 1'); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `Xc @ eigvecs[:, :2]` takes the dot product of each row with PC1 and PC2 &mdash; giving two
  numbers per point instead of five.
- the scatter is the compressed data. We colour by the *hidden* factor 1; the colour changes
  smoothly along PC1 &mdash; meaning PC1 essentially **rediscovered** our first hidden factor, which PCA
  was never told about.

**Answer here:**

1. The colour (hidden factor 1) lines up with one of the PC axes. In one sentence: what does that
   say PCA actually *found*?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Why we centre/scale first | - |
| What a covariance matrix shows | - |
| Eigenvectors as directions of variance | - |
| Reading a scree plot / elbow | - |
| Projecting data onto components | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what problem does PCA solve?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Check against sklearn

`from sklearn.decomposition import PCA` &mdash; fit `PCA(n_components=2)` on `Xc` and compare its
`explained_variance_ratio_` to your first two `explained` values. Fill it in.

In [ ]:
# from sklearn.decomposition import PCA
# your code here: fit PCA(2) on Xc and print pca.explained_variance_ratio_ next to explained[:2]


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 8

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/8/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 8 submission page](https://portal.latarak.com/course/cmsc173/lab/8/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.